# 03 - Feature Engineering

## Import

In [1]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import re
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import warnings
warnings.filterwarnings('ignore')

## Load dataset

In [2]:
SAVE_PATH = "../data/processed"

df_category = pl.read_parquet(f"{SAVE_PATH}/category.parquet")
df_seller = pl.read_parquet(f"{SAVE_PATH}/seller.parquet")
df_product = pl.read_parquet(f"{SAVE_PATH}/product.parquet")
df_price = pl.read_parquet(f"{SAVE_PATH}/price_offer.parquet")
df_review = pl.read_parquet(f"{SAVE_PATH}/review.parquet")
df_reviewer = pl.read_parquet(f"{SAVE_PATH}/reviewer.parquet")
df_service = pl.read_parquet(f"{SAVE_PATH}/service.parquet")
df_coupon = pl.read_parquet(f"{SAVE_PATH}/coupon.parquet")
df_offer_coupon = pl.read_parquet(f"{SAVE_PATH}/offer_coupon.parquet")
df_offer_service = pl.read_parquet(f"{SAVE_PATH}/offer_service.parquet")

In [3]:
PB_OUT = os.path.join(SAVE_PATH, "powerbi")
os.makedirs(PB_OUT, exist_ok=True)

## Mục tiêu phân tích

### Việt

---

**Mục tiêu:** Bóc tách cấu trúc doanh thu dự phóng (Dựa trên current_price $\times$ sold_quantity) qua 3 cấp độ danh mục để tìm ra 3 "Ngách thị trường ẩn" (Niche markets - Level 3) có đóng góp doanh số vượt trội hơn 50% so với trung bình Level 1 của chúng, phục vụ báo cáo mở rộng nguồn cung trước 10/04/2026

In [4]:
latest_price = df_price.sort("crawl_time").group_by("product_id").agg(
    pl.col("current_price").last(),
    pl.col("original_price").last(),
    pl.col("discount_percent").last(),
    pl.col("offer_id").last(),
    pl.col("crawl_time").last(),
)

df_cat_levels = df_category.with_columns(
    pl.col("category_path").str.split(">").alias("path_list")
).with_columns(
    pl.col("path_list").list.get(0, null_on_oob=True).str.strip_chars().alias("L1_Category"),
    pl.col("path_list").list.get(1, null_on_oob=True).str.strip_chars().alias("L2_Category"),
    pl.col("path_list").list.get(2, null_on_oob=True).str.strip_chars().alias("L3_Category")
).drop("path_list")

df_projected_rev = (
    latest_price.join(
        df_product.select(["product_id", "category_id", "sold_quantity"]), 
        on="product_id", how="inner"
    )
    .filter(pl.col("sold_quantity").is_not_null() & (pl.col("sold_quantity") > 0))
    .with_columns(
        (pl.col("current_price") * pl.col("sold_quantity")).alias("projected_revenue")
    )
)

pbi_niche_market = (
    df_projected_rev.join(df_cat_levels, on="category_id", how="left")
    .filter(pl.col("L3_Category").is_not_null()) 
    .group_by(["L1_Category", "L2_Category", "L3_Category"])
    .agg(
        pl.col("projected_revenue").sum().alias("l3_total_revenue"),
        pl.len().alias("sku_count")
    )
    .with_columns(
        pl.col("l3_total_revenue").mean().over("L1_Category").alias("l1_avg_l3_revenue")
    )
    .with_columns(
        (pl.col("l3_total_revenue") / pl.col("l1_avg_l3_revenue")).alias("outperform_ratio")
    )
)

pbi_niche_market.write_parquet(os.path.join(PB_OUT, "pbi_category_revenue_tree.parquet"))

**Mục tiêu:** Theo dõi sự biến động giá (current_price / original_price) theo chuỗi thời gian thực của 5 ngành hàng lớn nhất để xác định "Chu kỳ Sale" đặc thù của từng ngành, từ đó tư vấn thời điểm tung chiến dịch Flash Sale hiệu quả nhất trước ngày 30/04/2026.

In [5]:
cat_l1_mapping = df_category.with_columns(
    pl.col("category_path")
    .str.split(">")
    .list.get(0)
    .str.strip_chars()
    .alias("l1_category_name")
).select(["category_id", "l1_category_name"])

top5_l1_names = (
    df_product.join(cat_l1_mapping, on="category_id", how="left")
    .group_by("l1_category_name")
    .len()
    .sort("len", descending=True)
    .head(5)["l1_category_name"].to_list()
)

pbi_price_timeseries = (
    df_price
    .join(df_product.select(["product_id", "category_id"]), on="product_id", how="inner")
    .join(cat_l1_mapping, on="category_id", how="inner")
    .filter(pl.col("l1_category_name").is_in(top5_l1_names))
    
    .with_columns(
        pl.col("crawl_time").dt.date().alias("record_date"),
        (pl.col("current_price") / pl.col("original_price")).alias("price_ratio")
    )
    
    .group_by(["l1_category_name", "record_date"])
    .agg(
        pl.col("price_ratio").median().alias("median_price_ratio"),
        pl.col("discount_percent").median().alias("median_discount_pct"),
        pl.len().alias("active_listings") 
    )
    .sort(["l1_category_name", "record_date"])
)

pbi_price_timeseries.write_parquet(os.path.join(PB_OUT, "pbi_price_timeseries.parquet"))

**Mục tiêu:** Ứng dụng thuật toán Random Forest Classifier để phân loại tập sản phẩm thành 2 nhóm (Top 10% bán chạy và Phần còn lại). Từ đó trích xuất 3 biến số (giá, điểm đánh giá, số dịch vụ đính kèm...) có trọng số quyết định cao nhất đến doanh thu, nhằm tư vấn cho seller cách phân bổ nguồn lực vận hành trước ngày 15/04/2026.

In [6]:
df_services = (
    df_offer_service
    .group_by("offer_id")
    .agg(pl.len().alias("service_count"))
)

df_coupons = (
    df_offer_coupon
    .group_by("offer_id")
    .agg(pl.len().alias("coupon_count"))
)

df_latest_price = (
    df_price
    .join(df_services, on="offer_id", how="left")
    .join(df_coupons, on="offer_id", how="left")
    .with_columns([
        pl.col("service_count").fill_null(0),
        pl.col("coupon_count").fill_null(0) 
    ])
    .sort(["product_id", "crawl_time"], descending=True)
    .unique(subset=["product_id"], keep="first")
)

df_master = (
    df_product
    .join(df_latest_price, on="product_id", how="inner")
    .join(df_seller, on="seller_id", how="left")
)

df_master = df_master.with_columns(
    (pl.col("current_price") * pl.col("sold_quantity")).alias("revenue")
)

threshold_90 = df_master.select(pl.col("revenue").quantile(0.9)).item()

df_master = df_master.with_columns(
    pl.when(pl.col("revenue") >= threshold_90).then(1).otherwise(0).alias("is_top_10")
)

features = [
    "current_price", 
    "discount_percent", 
    "review_score", 
    "review_count", 
    "seller_rating", 
    "service_count",
    "coupon_count"
]

df_ml = df_master.with_columns([
    pl.col("review_score").fill_null(0),
    pl.col("review_count").fill_null(0),
    pl.col("seller_rating").fill_null(0),
    pl.col("discount_percent").fill_null(0)
]).drop_nulls(subset=features + ["is_top_10"])

X = df_ml.select(features).to_pandas()
y = df_ml.select("is_top_10").to_pandas()["is_top_10"]

print("Đang huấn luyện mô hình Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, max_depth=10)
rf_model.fit(X, y)

importances = rf_model.feature_importances_
df_importance = pl.DataFrame({
    "Feature_Name": features,
    "Importance_Score": importances
}).sort("Importance_Score", descending=True)

df_importance.write_parquet(os.path.join(PB_OUT, "rf_feature_importance.parquet"))

df_results = df_ml.with_columns(
    pl.Series("rf_prediction", rf_model.predict(X))
).select(
    ["product_id", "product_name", "revenue", "is_top_10"] + features
)

df_results.write_parquet(os.path.join(PB_OUT, "rf_product_analysis.parquet"))

Đang huấn luyện mô hình Random Forest...


**Mục tiêu:** Triển khai thuật toán NLP (Latent Dirichlet Allocation - LDA) trên tập dữ liệu tối thiểu 10.000 bình luận 1-2 sao để tự động gom cụm và định danh 3 nhóm nguyên nhân cốt lõi gây thất vọng (VD: Lỗi đóng gói, Giao hàng chậm), chiếm > 60% tổng phàn nàn, báo cáo trước 18/04/2026.

In [9]:
df_bad_reviews = df_review.filter(
    (pl.col("rating_score") <= 2) & 
    (pl.col("review_content").is_not_null()) & 
    (pl.col("review_content").str.len_chars() > 1) # Bỏ qua các review quá ngắn (vd: "tệ", "tốt")
)

print(f"Số lượng bình luận 1-2 sao cần phân tích: {df_bad_reviews.height}")

with open("../data/vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
    vietnamese_stopwords = f.read().splitlines()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_cleaned = df_bad_reviews.with_columns(
    pl.col("review_content").map_elements(clean_text, return_dtype=pl.Utf8).alias("cleaned_text")
)

docs = df_cleaned.get_column("cleaned_text").to_list()


vectorizer = CountVectorizer(max_df=0.90, min_df=5, stop_words=vietnamese_stopwords)
tf_matrix = vectorizer.fit_transform(docs)

lda_model = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=10)
lda_model.fit(tf_matrix)

feature_names = vectorizer.get_feature_names_out()
topic_keywords = {}

for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-16:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

topic_names = {
    "0": "Lỗi Vận hành & Trải nghiệm sản phẩm", 
    "1": "Lỗi Hình thức & Nội dung",
    "2": "Khiếu nại Dịch vụ & Quy trình Đổi trả",
    "3": "Sản phẩm không đúng Mô tả"
}

print("Đang phân loại chủ đề cho từng đánh giá...")
doc_topic_distributions = lda_model.transform(tf_matrix)
dominant_topics = doc_topic_distributions.argmax(axis=1)

df_final = df_cleaned.with_columns([
    pl.Series("topic_id", dominant_topics),
    pl.Series("topic_id_temp", dominant_topics)
      .cast(pl.String)
      .replace(topic_names)
      .alias("topic_name")
])

df_export = df_final.select([
    "review_id", "product_id", "rating_score", "review_content", "cleaned_text", "topic_name"
])

out_path = os.path.join(PB_OUT, "lda_bad_reviews_topics.parquet")
df_export.write_parquet(out_path)

Số lượng bình luận 1-2 sao cần phân tích: 18681
Topic 0: phẩm, sản, hàng, dụng, ok, sử, máy, chất, giao, gói, mua, đóng, ko, hành, ổn
Topic 1: sách, đọc, nội, dung, bìa, gói, mua, tiki, bọc, đóng, trang, hơi, ko, vọng, cũ
Topic 2: hàng, giao, tiki, mua, ko, shop, đơn, đổi, giá, tặng, tiền, đề, gửi, gọi, liên
Topic 3: ko, màu, mua, ng, mùi, bình, hơi, quảng, cáo, dán, nhựa, da, dây, nắp, đầu
Đang phân loại chủ đề cho từng đánh giá...



---

### Thương

**Mục tiêu:** TÍNH TỶ LỆ BIẾN ĐỘNG GIÁ PHỤC VỤ CHO MỤC TIÊU PHÂN TÍCH 1

In [8]:
# lấy giá mới nhất của mỗi product
df_price_latest = (
    df_price
    .filter(
        pl.col("product_id").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .sort(["product_id", "crawl_time"])
    .group_by("product_id")
    .tail(1)
    .select(["product_id", "current_price"])
)

df = (
    df_product
    .select(["product_id", "category_id"])
    .join(
        df_category.select(["category_id", "category_name"]),
        on="category_id",
        how="left"
    )
    .join(
        df_price_latest,
        on="product_id",
        how="left"
    )
    .filter(
        pl.col("category_name").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .with_columns(
        pl.col("current_price").log1p().alias("log_price")
    )
)

# Tính grand mean
grand_mean = df.select(pl.col("log_price").mean()).item()

# Tính SST = tổng biến thiên toàn bộ
sst = df.select(
    ((pl.col("log_price") - grand_mean) ** 2).sum().alias("sst")
).item()

# Tính mean theo từng ngành
group_means = (
    df.group_by("category_name")
    .agg(pl.col("log_price").mean().alias("group_mean"))
)

# Tính SSE = tổng biến thiên còn lại sau khi giải thích bởi ngành hàng
df_with_mean = df.join(group_means, on="category_name", how="left")

sse = df_with_mean.select(
    ((pl.col("log_price") - pl.col("group_mean")) ** 2).sum().alias("sse")
).item()

# R-squared
r2 = 1 - sse / sst

print(f"R-squared = {r2:.4f}")

R-squared = 0.6458
